---
## Stage 11: Model Training

**วัตถุประสงค์:** Train binary classifier ทำนายว่าคู่ profiles เป็นคนเดียวกันหรือไม่

**Input:** `train_loader`, `val_loader` (จาก Stage 10), `pos_weight`  
**Output:** `model.pt`, `training_history.csv`

| Sub-step | หน้าที่ |
|----------|--------|
| 11.1 | Model Architecture + Loss |
| 11.2 | Training Loop + Early Stopping |
| 11.3 | Training Curves Visualization |

### Step 11.1: Model Architecture Definition
MLP: Input → [Linear→BN→ReLU→Dropout] ×3 → Output + FocalLoss

In [ ]:
# --- 11.1 Model Architecture ---
import torch
import torch.nn as nn

class FocalLoss(nn.Module):
    """Focal Loss สำหรับ class imbalance — ลด weight ของ easy examples"""
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

class IdentityMLP(nn.Module):
    """MLP สำหรับ Identity Resolution: Input → 256 → 128 → 64 → 1"""
    def __init__(self, input_dim, hidden_dims=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)

# สร้าง model
input_dim = len(feature_cols)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = IdentityMLP(input_dim=input_dim).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("📊 Step 11.1: Model Architecture")
print("=" * 60)
print(f"  Device          : {device}")
print(f"  Input dim       : {input_dim}")
print(f"  Architecture    : {input_dim} → 256 → 128 → 64 → 1")
print(f"  Total params    : {total_params:,}")
print(f"  Trainable       : {trainable:,}")
print(model)
print(f"\n✅ Step 11.1 เสร็จ")

### Step 11.2: Training Loop + Early Stopping

In [ ]:
# --- 11.2 Training Loop ---
MAX_EPOCHS = 50
LR = 1e-3
PATIENCE = 5

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = FocalLoss(gamma=2.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

# Early Stopping
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0
history = []

print("📊 Step 11.2: Training Loop")
print("=" * 60)
print(f"  Epochs: {MAX_EPOCHS}, LR: {LR}, Patience: {PATIENCE}")
print(f"  Loss: FocalLoss(gamma=2.0)")
print("-" * 60)

for epoch in range(MAX_EPOCHS):
    # --- Train ---
    model.train()
    train_losses, train_correct, train_total = [], 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == y_batch).sum().item()
        train_total += len(y_batch)
    
    # --- Validate ---
    model.eval()
    val_losses, val_correct, val_total = [], 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_losses.append(loss.item())
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == y_batch).sum().item()
            val_total += len(y_batch)
    
    scheduler.step()
    
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)
    
    history.append({'epoch': epoch+1, 'train_loss': train_loss, 'val_loss': val_loss,
                    'train_acc': train_acc, 'val_acc': val_acc})
    
    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'model.pt'))
    else:
        patience_counter += 1
    
    marker = ' ← BEST' if patience_counter == 0 else ''
    print(f"  Epoch {epoch+1:3d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
          f"train_acc={train_acc:.3f} val_acc={val_acc:.3f}{marker}")
    
    if patience_counter >= PATIENCE:
        print(f"\n  ⏹️ Early stopping at epoch {epoch+1} (patience={PATIENCE})")
        break

# Load best model
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'model.pt'), weights_only=True))

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)

print(f"\n  🏆 Best epoch: {best_epoch} (val_loss={best_val_loss:.4f})")
print(f"  💾 Saved: model.pt, training_history.csv")
print(f"\n✅ Step 11.2 เสร็จ")

### Step 11.3: Training Curves Visualization

In [ ]:
# --- 11.3 Training Curves ---
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_df['epoch'], history_df['train_loss'], 'b-', label='Train Loss')
ax1.plot(history_df['epoch'], history_df['val_loss'], 'r-', label='Val Loss')
ax1.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss Curves')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_df['epoch'], history_df['train_acc'], 'b-', label='Train Acc')
ax2.plot(history_df['epoch'], history_df['val_acc'], 'r-', label='Val Acc')
ax2.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy Curves')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*60}")
print(f"✅ Stage 11 COMPLETE — Model trained & saved")
print(f"{'='*60}")